In [56]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)

from fairness_metrics import (
    demographic_parity, disparate_impact_ratio, equalized_odds_tpr,
    mean_prediction_difference, residual_error_parity, r2_parity
)
warnings.filterwarnings("ignore")


In [57]:
data = pd.read_csv("../preprocessing/data/cleaned_laptop_data.csv")

artifact_dir = Path("../notebooks/artifacts")

clf_model = joblib.load(artifact_dir / "best_classifier_model.pkl")
clf_preprocessor = joblib.load(artifact_dir / "clf_preprocessor.pkl")
clf_features = joblib.load(artifact_dir / "classification_features.pkl")

reg_model = joblib.load(artifact_dir / "best_regressor_model.pkl")
reg_preprocessor = joblib.load(artifact_dir / "reg_preprocessor.pkl")
reg_features = joblib.load(artifact_dir / "regression_features.pkl")

print(data.shape)

(19545, 53)


In [58]:
classification_fairness_features = [
    "performance_score",
    "upgrade_pressure",
    "total_urgency",
    "budget_class",
    "user_profile"
]

regression_fairness_features = [                        
    "performance_score",
    "upgrade_pressure",
    "budget_class",
    "total_urgency",
    "type_name"
]

print("Classification fairness features:", classification_fairness_features)
print("Regression fairness features:", regression_fairness_features)

Classification fairness features: ['performance_score', 'upgrade_pressure', 'total_urgency', 'budget_class', 'user_profile']
Regression fairness features: ['performance_score', 'upgrade_pressure', 'budget_class', 'total_urgency', 'type_name']


In [59]:
protected_group_clf = "budget_class"
protected_group_reg = "budget_class"

print("Classification protected group:", protected_group_clf)
print("Regression protected group:", protected_group_reg)
display(data[protected_group_clf].value_counts()) 

Classification protected group: budget_class
Regression protected group: budget_class


budget_class
low       6515
medium    6515
high      6515
Name: count, dtype: int64

In [60]:
X_clf = data[clf_features].copy()
y_clf = data["upgrade_first"].copy()

X_clf_p = clf_preprocessor.transform(X_clf)
y_clf_pred = clf_model.predict(X_clf_p)

clf_results = data.copy()
clf_results["y_true"] = y_clf
clf_results["y_pred"] = y_clf_pred

clf_results[[protected_group_clf, "y_true", "y_pred"]].head(20)

,budget_class,y_true,y_pred
0,low,Storage,Storage
1,medium,Storage,Storage
2,high,Storage,Storage
3,low,Storage,Storage
4,medium,Storage,Storage
5,high,Storage,Storage
6,low,RAM,RAM
7,medium,RAM,RAM
8,high,RAM,RAM
9,low,RAM,RAM


In [61]:
X_reg = data[reg_features].copy()
y_reg = data["recommended_upgrade_cost_est"].copy()

X_reg_p = reg_preprocessor.transform(X_reg)
y_reg_pred = reg_model.predict(X_reg_p)

reg_results = data.copy()
reg_results["y_true"] = y_reg
reg_results["y_pred"] = y_reg_pred
reg_results["residual"] = reg_results["y_true"] - reg_results["y_pred"]

reg_results[[protected_group_reg, "y_true", "y_pred", "residual"]].head()

,budget_class,y_true,y_pred,residual
0,low,43.72,43.699181,2.081905e-02
1,medium,43.72,43.699181,2.081905e-02
2,high,43.72,43.699181,2.081905e-02
3,low,43.72,43.720000,-3.552714e-14
4,medium,43.72,43.654420,6.558000e-02


In [62]:
high_upgrade_classes = ["GPU", "CPU", "Storage+RAM"]  

clf_results["positive_true"] = clf_results["y_true"].isin(high_upgrade_classes).astype(int)
clf_results["positive_pred"] = clf_results["y_pred"].isin(high_upgrade_classes).astype(int)

clf_results[["y_true", "y_pred", "positive_true", "positive_pred"]].head()

,y_true,y_pred,positive_true,positive_pred
0,Storage,Storage,0,0
1,Storage,Storage,0,0
2,Storage,Storage,0,0
3,Storage,Storage,0,0
4,Storage,Storage,0,0


In [63]:
dp_clf = demographic_parity(clf_results, protected_group_clf)
dir_clf = disparate_impact_ratio(clf_results, protected_group_clf)
eo_clf = equalized_odds_tpr(clf_results, protected_group_clf)

print("=== Classification Fairness (Original Model) ===")
print("\nDemographic Parity:")
display(dp_clf)

print("\nDisparate Impact Ratio:")
print(dir_clf)

print("\nEqualized Odds (TPR by group):")
display(eo_clf)

=== Classification Fairness (Original Model) ===

Demographic Parity:


budget_class
high      0.194781
low       0.016424
medium    0.122026
Name: positive_pred, dtype: float64


Disparate Impact Ratio:
0.08431836091410559

Equalized Odds (TPR by group):


,budget_class,TPR
0,high,0.997642
1,low,1.000000
2,medium,1.000000


In [64]:
print("=== Classification Performance (Original Model) ===")
print("Accuracy :", accuracy_score(y_clf, y_clf_pred))
print("Precision:", precision_score(y_clf, y_clf_pred, average="weighted", zero_division=0))
print("Recall   :", recall_score(y_clf, y_clf_pred, average="weighted", zero_division=0))
print("F1       :", f1_score(y_clf, y_clf_pred, average="weighted", zero_division=0))

=== Classification Performance (Original Model) ===
Accuracy : 0.9996930161166538
Precision: 0.9996930251861993
Recall   : 0.9996930161166538
F1       : 0.9996929861952938


In [65]:
mpd_reg = mean_prediction_difference(reg_results, protected_group_reg)
rep_reg = residual_error_parity(reg_results, protected_group_reg)
r2p_reg = r2_parity(reg_results, protected_group_reg)

print("=== Regression Fairness (Original Model) ===")
print("\nMean Prediction Difference:")
display(mpd_reg)

print("\nResidual Error Parity (lower is better):")
display(rep_reg)

print("\nR² Performance Parity:")
display(r2p_reg) 

=== Regression Fairness (Original Model) ===

Mean Prediction Difference:


budget_class
high      104.086902
low        35.400421
medium     73.500068
Name: y_pred, dtype: float64


Residual Error Parity (lower is better):


budget_class
high      2.572116
low       1.315925
medium    1.575106
Name: residual, dtype: float64


R² Performance Parity:


,budget_class,R2
0,high,0.992718
1,low,0.982305
2,medium,0.996461


In [66]:
print("=== Regression Performance (Original Model) ===")
print("MAE :", mean_absolute_error(y_reg, y_reg_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_reg, y_reg_pred)))
print("R2  :", r2_score(y_reg, y_reg_pred))

=== Regression Performance (Original Model) ===
MAE : 1.8210492447593516
RMSE: 10.47082839813646
R2  : 0.9936979954757801


In [89]:
protected_group_clf = "budget_class"

high_upgrade_classes = ["GPU", "CPU", "Storage+RAM"]

proxy_features_to_remove = [
    "budget_class",
    "price",
    "price_range",
    "budget_level",
    "price_performance",
    "price_per_performance",
    "budget_level"
]

fair_clf_features = [f for f in clf_features if f not in proxy_features_to_remove]

X = data[fair_clf_features].copy()
y = data["upgrade_first"].copy()
groups = data[protected_group_clf].copy()

X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    X, y, groups,
    test_size=0.2,
    random_state=42,
    stratify=y
)

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]

fair_clf_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X_train_p = fair_clf_preprocessor.fit_transform(X_train)
X_test_p = fair_clf_preprocessor.transform(X_test)

combo = group_train.astype(str) + "_" + y_train.astype(str)
sample_weights = compute_sample_weight(class_weight="balanced", y=combo)

fair_clf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

fair_clf_model.fit(X_train_p, y_train, sample_weight=sample_weights)

y_pred = fair_clf_model.predict(X_test_p)

clf_test_results_fair = X_test.copy()
clf_test_results_fair["budget_class"] = group_test.values
clf_test_results_fair["y_true"] = y_test.values
clf_test_results_fair["y_pred"] = y_pred

clf_test_results_fair["positive_true"] = clf_test_results_fair["y_true"].isin(high_upgrade_classes).astype(int)

proba_df = pd.DataFrame(
    fair_clf_model.predict_proba(X_test_p),
    columns=fair_clf_model.classes_
)

existing_high_classes = [c for c in high_upgrade_classes if c in proba_df.columns]
clf_test_results_fair["positive_score"] = proba_df[existing_high_classes].sum(axis=1).values

thresholds = {
    "low": 0.58,
    "medium": 0.55,
    "high": 0.50
}

clf_test_results_fair["positive_pred"] = clf_test_results_fair.apply(
    lambda row: int(row["positive_score"] >= thresholds[row["budget_class"]]),
    axis=1
)

print("=== Classification Fairness After Mitigation ===")
display(demographic_parity(clf_test_results_fair, "budget_class"))

dir_clf_fair = disparate_impact_ratio(clf_test_results_fair, "budget_class")
print("DIR:", dir_clf_fair)

display(equalized_odds_tpr(clf_test_results_fair, "budget_class"))

print("\n=== Binary Fairness Decision Performance ===")
print("Accuracy :", accuracy_score(clf_test_results_fair["positive_true"], clf_test_results_fair["positive_pred"]))
print("Precision:", precision_score(clf_test_results_fair["positive_true"], clf_test_results_fair["positive_pred"], zero_division=0))
print("Recall   :", recall_score(clf_test_results_fair["positive_true"], clf_test_results_fair["positive_pred"], zero_division=0))
print("F1       :", f1_score(clf_test_results_fair["positive_true"], clf_test_results_fair["positive_pred"], zero_division=0))

print("\n=== Multiclass Classification Performance ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1       :", f1_score(y_test, y_pred, average="weighted", zero_division=0))

=== Classification Fairness After Mitigation ===


budget_class
high      0.211783
low       0.131184
medium    0.156179
Name: positive_pred, dtype: float64

DIR: 0.6194271285409926


,budget_class,TPR
0,high,0.981273
1,low,1.000000
2,medium,0.992908



=== Binary Fairness Decision Performance ===
Accuracy : 0.9424405218726016
Precision: 0.6615146831530139
Recall   : 0.9861751152073732
F1       : 0.7918593894542091

=== Multiclass Classification Performance ===
Accuracy : 0.9130212330519314
Precision: 0.9498350620789139
Recall   : 0.9130212330519314
F1       : 0.9211485140566583


In [86]:

fair_reg_features = [f for f in reg_features if f not in ["budget_class", "price"]]

X_reg_fair = data[fair_reg_features].copy()
y_reg_fair = data["recommended_upgrade_cost_est"].copy()

X_train_reg_fair, X_test_reg_fair, y_train_reg_fair, y_test_reg_fair = train_test_split(
    X_reg_fair,
    y_reg_fair,
    test_size=0.2,
    random_state=42
)

fair_reg_num = [c for c in X_train_reg_fair.columns if pd.api.types.is_numeric_dtype(X_train_reg_fair[c])]
fair_reg_cat = [c for c in X_train_reg_fair.columns if X_train_reg_fair[c].dtype == "object"]

fair_reg_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), fair_reg_num),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), fair_reg_cat),
])

fair_reg_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

X_train_reg_fair_p = fair_reg_preprocessor.fit_transform(X_train_reg_fair)
X_test_reg_fair_p = fair_reg_preprocessor.transform(X_test_reg_fair)

fair_reg_model.fit(X_train_reg_fair_p, y_train_reg_fair)
y_reg_fair_pred = fair_reg_model.predict(X_test_reg_fair_p)

reg_test_results_fair = X_test_reg_fair.copy()
reg_test_results_fair["y_true"] = y_test_reg_fair.values
reg_test_results_fair["y_pred"] = y_reg_fair_pred
reg_test_results_fair["residual"] = reg_test_results_fair["y_true"] - reg_test_results_fair["y_pred"]

reg_test_results_fair[protected_group_reg] = data.loc[reg_test_results_fair.index, protected_group_reg]

mpd_reg_fair = mean_prediction_difference(reg_test_results_fair, protected_group_reg)
rep_reg_fair = residual_error_parity(reg_test_results_fair, protected_group_reg)
r2p_reg_fair = r2_parity(reg_test_results_fair, protected_group_reg)

print("=== Regression Fairness (Fair Model) ===")
display(mpd_reg_fair)
display(rep_reg_fair)
display(r2p_reg_fair)

print("\n=== Regression Performance (Fair Model) ===")
print("MAE :", mean_absolute_error(y_test_reg_fair, y_reg_fair_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test_reg_fair, y_reg_fair_pred)))
print("R2  :", r2_score(y_test_reg_fair, y_reg_fair_pred))

=== Regression Fairness (Fair Model) ===


budget_class
high      108.066297
low        35.443174
medium     73.658865
Name: y_pred, dtype: float64

budget_class
high      2.902401
low       1.468449
medium    1.213160
Name: residual, dtype: float64

,budget_class,R2
0,high,0.984892
1,low,0.915004
2,medium,0.991221



=== Regression Performance (Fair Model) ===
MAE : 1.8622012535176524
RMSE: 17.091827731282518
R2  : 0.9837958086399642


In [70]:
classification_comparison = pd.DataFrame({
    "Metric": ["DIR", "Accuracy", "F1"],
    "Original": [
        dir_clf,
        accuracy_score(y_clf, y_clf_pred),
        f1_score(y_clf, y_clf_pred, average="weighted", zero_division=0)
    ],
    "Fair": [
        dir_clf_fair,
        accuracy_score(y_test, y_pred),
        f1_score(y_test, y_pred, average="weighted", zero_division=0)
    ]
})

display(classification_comparison)


,Metric,Original,Fair
0,DIR,0.084318,0.867593
1,Accuracy,0.999693,0.913021
2,F1,0.999693,0.921149


In [ ]:
regression_comparison = pd.DataFrame({
    "Metric": ["Overall MAE", "Overall RMSE", "Overall R2"],
    "Original": [
        mean_absolute_error(y_reg, y_reg_pred),
        np.sqrt(mean_squared_error(y_reg, y_reg_pred)),
        r2_score(y_reg, y_reg_pred)
    ],
    "Fair": [
        mean_absolute_error(y_test_reg_fair, y_reg_fair_pred),
        np.sqrt(mean_squared_error(y_test_reg_fair, y_reg_fair_pred)),
        r2_score(y_test_reg_fair, y_reg_fair_pred)
    ]
})

display(regression_comparison)

,Metric,Original,Fair
0,Overall MAE,1.821049,1.862201
1,Overall RMSE,10.470828,17.091828
2,Overall R2,0.993698,0.983796


In [90]:
from BiasFairness import ClassificationBiasFairness
import pandas as pd

cbf = ClassificationBiasFairness()
budget_groups = clf_test_results_fair["budget_class"].dropna().unique().tolist()

print("=== BiasFairness: Classification Model ===")
bias_clf = cbf.check_feature_bias_across_groups(
    df=clf_test_results_fair,
    group_column="budget_class",
    groups=budget_groups,
    outcome_col="positive_pred"
)
print("Feature Bias (DIR & DP):")
for res in bias_clf:
    print(res)

eo_clf = cbf.check_eo_across_groups(
    df=clf_test_results_fair,
    group_column="budget_class",
    groups=budget_groups,
    actual_col="positive_true",
    decision_col="positive_pred"
)
print("\nEqualized Odds:")
for res in eo_clf:
    print(res)


=== BiasFairness: Classification Model ===
Feature Bias (DIR & DP):
{'groups': 'medium vs low', 'DIR': 0.84, 'DIR_fairness': 'fair', 'DP': 0.025, 'DP_fairness': 'fair'}
{'groups': 'medium vs high', 'DIR': 0.737, 'DIR_fairness': 'bias', 'DP': 0.056, 'DP_fairness': 'bias'}
{'groups': 'low vs high', 'DIR': 0.619, 'DIR_fairness': 'bias', 'DP': 0.081, 'DP_fairness': 'bias'}

Equalized Odds:
{'groups': 'medium vs low', 'group_a_metrics': {'TPR': np.float64(0.993), 'FPR': np.float64(0.056)}, 'group_b_metrics': {'TPR': np.float64(1.0), 'FPR': np.float64(0.114)}, 'TPR_diff': np.float64(0.007), 'FPR_diff': np.float64(0.058), 'EO_fairness': 'bias'}
{'groups': 'medium vs high', 'group_a_metrics': {'TPR': np.float64(0.993), 'FPR': np.float64(0.056)}, 'group_b_metrics': {'TPR': np.float64(0.981), 'FPR': np.float64(0.004)}, 'TPR_diff': np.float64(0.012), 'FPR_diff': np.float64(0.052), 'EO_fairness': 'bias'}
{'groups': 'low vs high', 'group_a_metrics': {'TPR': np.float64(1.0), 'FPR': np.float64(0.114)

In [ ]:
from BiasFairness import RegressionBiasFairness

rbf = RegressionBiasFairness(threshold_type="relative", mpd_threshold=0.05, mae_threshold=0.05)
budget_groups_reg = reg_test_results_fair[protected_group_reg].dropna().unique().tolist()

print("=== BiasFairness: Regression Model ===")
mae_reg = rbf.check_mae_across_groups(
    df=reg_test_results_fair,
    group_column=protected_group_reg,
    groups=budget_groups_reg,
    actual_col="y_true",
    predicted_col="y_pred"
)
print("MAE Parity:")
for res in mae_reg:
    print(res)

r2_reg = rbf.check_r2_parity_across_groups(
    df=reg_test_results_fair,
    group_column=protected_group_reg,
    groups=budget_groups_reg,
    actual_col="y_true",
    predicted_col="y_pred",
    r2_threshold=0.1
)
print("\nR2 Parity:")
for res in r2_reg:
    print(res)


=== BiasFairness: Regression Model ===
MAE Parity:
{'groups': 'low vs medium', 'MAE_a': 1.468, 'MAE_b': 1.213, 'MAE_diff': 0.255, 'MAE_fairness': 'fair', 'relative_MAE_diff': 0.0035}
{'groups': 'low vs high', 'MAE_a': 1.468, 'MAE_b': 2.902, 'MAE_diff': 1.434, 'MAE_fairness': 'fair', 'relative_MAE_diff': 0.0198}
{'groups': 'medium vs high', 'MAE_a': 1.213, 'MAE_b': 2.902, 'MAE_diff': 1.689, 'MAE_fairness': 'fair', 'relative_MAE_diff': 0.0233}

R2 Parity:
{'groups': 'low vs medium', 'R2_a': 0.915, 'R2_b': 0.991, 'R2_diff': 0.076, 'R2_fairness': 'fair'}
{'groups': 'low vs high', 'R2_a': 0.915, 'R2_b': 0.985, 'R2_diff': 0.07, 'R2_fairness': 'fair'}
{'groups': 'medium vs high', 'R2_a': 0.991, 'R2_b': 0.985, 'R2_diff': 0.006, 'R2_fairness': 'fair'}


In [91]:
from itertools import product
import numpy as np

def group_rates_for_thresholds(df, thresholds):
    temp = df.copy()

    temp["positive_pred"] = temp.apply(
        lambda row: int(row["positive_score"] >= thresholds[row["budget_class"]]),
        axis=1
    )

    rates = []

    for group, gdf in temp.groupby("budget_class"):
        tp = ((gdf["positive_true"] == 1) & (gdf["positive_pred"] == 1)).sum()
        fn = ((gdf["positive_true"] == 1) & (gdf["positive_pred"] == 0)).sum()
        fp = ((gdf["positive_true"] == 0) & (gdf["positive_pred"] == 1)).sum()
        tn = ((gdf["positive_true"] == 0) & (gdf["positive_pred"] == 0)).sum()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        selection_rate = gdf["positive_pred"].mean()

        rates.append({
            "group": group,
            "TPR": tpr,
            "FPR": fpr,
            "selection_rate": selection_rate
        })

    rates = pd.DataFrame(rates)

    dir_value = rates["selection_rate"].min() / rates["selection_rate"].max()

    eo_gap = max(
        rates["TPR"].max() - rates["TPR"].min(),
        rates["FPR"].max() - rates["FPR"].min()
    )

    dp_gap = rates["selection_rate"].max() - rates["selection_rate"].min()

    return temp, rates, dir_value, eo_gap, dp_gap


best = None
threshold_values = np.arange(0.35, 0.76, 0.01)

for low_t, medium_t, high_t in product(threshold_values, threshold_values, threshold_values):

    thresholds = {
        "low": low_t,
        "medium": medium_t,
        "high": high_t
    }

    temp, rates, dir_value, eo_gap, dp_gap = group_rates_for_thresholds(
        clf_test_results_fair,
        thresholds
    )

    binary_f1 = f1_score(
        temp["positive_true"],
        temp["positive_pred"],
        zero_division=0
    )

    # Main goal:
    # DIR >= 0.8 and EO gap <= 0.05 if possible
    # Then choose the best F1
    if dir_value >= 0.8 and eo_gap <= 0.05:
        score = binary_f1

        if best is None or score > best["score"]:
            best = {
                "thresholds": thresholds,
                "rates": rates,
                "dir": dir_value,
                "eo_gap": eo_gap,
                "dp_gap": dp_gap,
                "f1": binary_f1,
                "score": score,
                "temp": temp
            }

# Backup: if no perfect threshold set exists, choose best combined trade-off
if best is None:
    for low_t, medium_t, high_t in product(threshold_values, threshold_values, threshold_values):

        thresholds = {
            "low": low_t,
            "medium": medium_t,
            "high": high_t
        }

        temp, rates, dir_value, eo_gap, dp_gap = group_rates_for_thresholds(
            clf_test_results_fair,
            thresholds
        )

        binary_f1 = f1_score(
            temp["positive_true"],
            temp["positive_pred"],
            zero_division=0
        )

        score = (dir_value * 0.45) - (eo_gap * 0.35) - (dp_gap * 0.20) + (binary_f1 * 0.10)

        if best is None or score > best["score"]:
            best = {
                "thresholds": thresholds,
                "rates": rates,
                "dir": dir_value,
                "eo_gap": eo_gap,
                "dp_gap": dp_gap,
                "f1": binary_f1,
                "score": score,
                "temp": temp
            }

clf_test_results_fair = best["temp"].copy()

print("Best thresholds:")
print(best["thresholds"])

print("\nGroup rates:")
display(best["rates"])

print("\nDIR:", round(best["dir"], 3))
print("EO gap:", round(best["eo_gap"], 3))
print("DP gap:", round(best["dp_gap"], 3))
print("Binary F1:", round(best["f1"], 3))

print("\n=== Classification Fairness After Threshold Search ===")
display(demographic_parity(clf_test_results_fair, "budget_class"))

dir_clf_fair = disparate_impact_ratio(clf_test_results_fair, "budget_class")
print("DIR:", dir_clf_fair)

display(equalized_odds_tpr(clf_test_results_fair, "budget_class"))

Best thresholds:
{'low': np.float64(0.45000000000000007), 'medium': np.float64(0.4700000000000001), 'high': np.float64(0.6400000000000002)}

Group rates:


,group,TPR,FPR,selection_rate
0,high,0.865169,0.000000,0.183917
1,low,1.000000,0.167431,0.183658
2,medium,0.992908,0.087436,0.184230



DIR: 0.997
EO gap: 0.167
DP gap: 0.001
Binary F1: 0.689

=== Classification Fairness After Threshold Search ===


budget_class
high      0.183917
low       0.183658
medium    0.184230
Name: positive_pred, dtype: float64

DIR: 0.9968935285443697


,budget_class,TPR
0,high,0.865169
1,low,1.000000
2,medium,0.992908


In [ ]:
joblib.dump(fair_clf_model, artifact_dir / "fair_classifier.pkl")
joblib.dump(fair_clf_preprocessor, artifact_dir / "fair_clf_preprocessor.pkl")

joblib.dump(fair_reg_model, artifact_dir / "fair_regressor.pkl")
joblib.dump(fair_reg_preprocessor, artifact_dir / "fair_reg_preprocessor.pkl")

print("Fair models saved.") 

Fair models saved.
